# Taxi-Graph Analytics with GraphFrames

The taxi data is modelled as a directed weighted graph:
- **Vertices** = NYC geographic areas (pickup/dropoff zones)
- **Edges** = taxi trips (pickup area → dropoff area), weighted by `trip_count`, `avg_fare`, `avg_duration_min`, `avg_distance`

---
## 1. Environment setup

GraphFrames requires Java 17 and PySpark 3.5. Run these cells first.

In [53]:
# Install everything needed
!apt-get install -y openjdk-11-jdk-headless -q
!pip install pyspark==3.5.3 plotly pandas -q
!pip install graphframes==0.6.0 --no-deps -q
print('Installation done.')

zsh:1: command not found: apt-get
Installation done.


In [54]:
import os, subprocess

# Find actual Java path and set JAVA_HOME
java_bin = subprocess.run(['which', 'java'], capture_output=True, text=True).stdout.strip()
java_real = subprocess.run(['readlink', '-f', java_bin], capture_output=True, text=True).stdout.strip()
os.environ['JAVA_HOME'] = java_real.replace('/bin/java', '')
print(f'JAVA_HOME: {os.environ["JAVA_HOME"]}')
print(subprocess.run(['java', '-version'], capture_output=True, text=True).stderr.strip())

JAVA_HOME: /Users/jennykerrcubelo/miniconda3/envs/bda_spark/lib/jvm
openjdk version "17.0.18" 2026-01-20 LTS
OpenJDK Runtime Environment Zulu17.64+17-CA (build 17.0.18+8-LTS)
OpenJDK 64-Bit Server VM Zulu17.64+17-CA (build 17.0.18+8-LTS, mixed mode, sharing)


In [56]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import NumericType

spark = (
    SparkSession.builder
    .appName("TaxiGraphAnalytics")
    .master("local[*]")
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.2-s_2.12")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

# Import AFTER SparkSession so JAR is already loaded
from graphframes import GraphFrame

print(f"Spark version: {spark.version}")

Spark version: 3.5.3


In [57]:
print(f"Spark version: {spark.version}")
import pyspark
print(f"PySpark version: {pyspark.__version__}")
!python3 -c "import pyspark; print(pyspark.__version__)"

Spark version: 3.5.3
PySpark version: 3.5.3
3.5.3


---
## 2. Load and clean the dataset


In [58]:
file_location = "train.csv"
file_type = "csv"

df = spark.read.format(file_type) \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .option("sep", ",") \
    .load(file_location)

print(f"Raw rows: {df.count():,}")
df.printSchema()

Raw rows: 50,000
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [59]:
df = df.drop("VendorID", "store_and_fwd_flag", "payment_type")

In [60]:
date_format = "yyyy-MM-dd HH:mm:ss"

df = df.withColumn("pickup_ts",  F.to_timestamp(F.col("tpep_pickup_datetime"),  date_format))
df = df.withColumn("dropoff_ts", F.to_timestamp(F.col("tpep_dropoff_datetime"), date_format))

df = df.withColumn(
    "target_time",
    F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")
)
df = df.withColumn("pickup_hour", F.hour(F.col("pickup_ts")))
df = df.filter((F.col("target_time") > 0) & (F.col("target_time") < 14400))

df = df.drop("tpep_pickup_datetime", "passenger_count", "total_amount")

In [61]:
EARTH_RADIUS = 6371.0

lat1 = F.radians(F.col("pickup_latitude"))
lon1 = F.radians(F.col("pickup_longitude"))
lat2 = F.radians(F.col("dropoff_latitude"))
lon2 = F.radians(F.col("dropoff_longitude"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
haversine_dist = EARTH_RADIUS * c

df = df.withColumn(
    "trip_distance",
    F.when(
        (F.col("trip_distance") <= 0) | (F.col("trip_distance").isNull()),
        haversine_dist
    ).otherwise(F.col("trip_distance"))
)

In [62]:
NYC_HOTSPOTS = {
    "JFK_Airport":       (-73.7781, 40.6413),
    "LaGuardia_Airport": (-73.8740, 40.7769),
    "Times_Square":      (-73.9851, 40.7589),
    "Empire_State":      (-73.9857, 40.7484),
    "Central_Park":      (-73.9654, 40.7829),
    "Wall_Street":       (-74.0113, 40.7074),
    "Brooklyn_Bridge":   (-73.9969, 40.7061),
    "Williamsburg":      (-73.9571, 40.7081),
    "Astoria":           (-73.9235, 40.7644),
    "Long_Island_City":  (-73.9388, 40.7505),
    "Yankee_Stadium":    (-73.9262, 40.8296),
    "Harlem":            (-73.9465, 40.8116),
}

hotspots_data = [(name, coords[0], coords[1]) for name, coords in NYC_HOTSPOTS.items()]
hotspots_df = spark.createDataFrame(hotspots_data, ["area_name", "h_lon", "h_lat"])

def match_areas(input_df, lat_col, lon_col, final_name):
    temp_id_df = input_df.withColumn("_internal_id", F.monotonically_increasing_id())
    joined_df = temp_id_df.crossJoin(F.broadcast(hotspots_df)) \
        .withColumn("dist", F.sqrt(F.pow(F.col(lon_col) - F.col("h_lon"), 2) +
                                   F.pow(F.col(lat_col) - F.col("h_lat"), 2)))
    window_spec = Window.partitionBy("_internal_id").orderBy("dist")
    return joined_df.withColumn("rn", F.row_number().over(window_spec)) \
        .filter(F.col("rn") == 1) \
        .drop("h_lon", "h_lat", "dist", "rn", "_internal_id") \
        .withColumnRenamed("area_name", final_name)

df = match_areas(df, "pickup_latitude",  "pickup_longitude",  "area_pickup")
df = match_areas(df, "dropoff_latitude", "dropoff_longitude", "area_dropoff")

cols_to_remove = ["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"]
df = df.drop(*cols_to_remove)

In [63]:
df = df.withColumn("fare_amount", F.abs(F.col("fare_amount"))) \
       .withColumn("extra",       F.abs(F.col("extra"))) \
       .withColumn("tip_amount",  F.abs(F.col("tip_amount"))) \
       .withColumn("mta_tax",               F.lit(0.5)) \
       .withColumn("improvement_surcharge", F.lit(0.3))

df = df.filter((F.col("target_time") > 0) & (F.col("fare_amount") > 0))

In [64]:
excluded_cols = ["fare_amount", "target_time", "tip_amount", "RatecodeID", "pickup_hour"]
numeric_cols = [f.name for f in df.schema.fields
                if isinstance(f.dataType, NumericType) and f.name not in excluded_cols]

learned_parameters = {}
for col_name in numeric_cols:
    quantiles = df.approxQuantile(col_name, [0.25, 0.75], 0.05)
    if len(quantiles) < 2: continue
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    mean_val = df.select(F.mean(col_name)).collect()[0][0]
    learned_parameters[col_name] = {"upper_bound": upper_bound, "mean": mean_val}
    df = df.withColumn(
        col_name,
        F.when(F.col(col_name) > upper_bound, mean_val).otherwise(F.col(col_name))
    )
    print(f"Processed {col_name}: bound={upper_bound:.2f}, mean={mean_val:.2f}")

Processed trip_distance: bound=5.32, mean=6.55
Processed extra: bound=1.25, mean=0.31
Processed mta_tax: bound=0.50, mean=0.50
Processed tolls_amount: bound=0.00, mean=0.31
Processed improvement_surcharge: bound=0.30, mean=0.30


In [65]:
df = df.drop("tpep_dropoff_datetime", "pickup_ts", "dropoff_ts", "mta_tax", "improvement_surcharge")

df.cache()
print(f"Final rows: {df.count():,}")
print("Final columns:", df.columns)
df.show(5)

Final rows: 49,863
Final columns: ['trip_distance', 'RatecodeID', 'fare_amount', 'extra', 'tip_amount', 'tolls_amount', 'target_time', 'pickup_hour', 'area_pickup', 'area_dropoff']
+-----------------+----------+-----------+-----+----------+-----------------+-----------+-----------+------------+-----------------+
|    trip_distance|RatecodeID|fare_amount|extra|tip_amount|     tolls_amount|target_time|pickup_hour| area_pickup|     area_dropoff|
+-----------------+----------+-----------+-----+----------+-----------------+-----------+-----------+------------+-----------------+
|              1.2|         1|       11.0|  0.0|       1.0|              0.0|        972|         14|Empire_State|     Times_Square|
|             1.05|         1|        6.0|  0.0|       1.7|              0.0|        358|          7|Empire_State|     Times_Square|
|              1.5|         1|        9.0|  0.5|      2.05|              0.0|        721|         20|Empire_State|     Empire_State|
|6.545169073170687|  

In [66]:
# Alias for graph construction — same df, clearer name
taxi = df
print(f"Taxi rows ready for graph: {taxi.count():,}")

Taxi rows ready for graph: 49,863


---
## 3. Build the GraphFrame

- **Vertices** = unique areas, enriched with pickup/dropoff volume stats
- **Edges** = (area_pickup → area_dropoff) aggregated: `trip_count`, `avg_fare`, `avg_duration_min`, `avg_distance`


In [67]:
pickup_stats = (
    taxi.groupBy("area_pickup")
        .agg(
            F.count("*").alias("total_pickups"),
            F.round(F.avg("fare_amount"), 2).alias("avg_pickup_fare")
        )
        .withColumnRenamed("area_pickup", "id")
)
dropoff_stats = (
    taxi.groupBy("area_dropoff")
        .agg(F.count("*").alias("total_dropoffs"))
        .withColumnRenamed("area_dropoff", "id")
)

vertices = pickup_stats.join(dropoff_stats, on="id", how="outer").fillna(0)

print(f"Vertices: {vertices.count()}")
vertices.show(truncate=False)

Vertices: 12
+-----------------+-------------+---------------+--------------+
|id               |total_pickups|avg_pickup_fare|total_dropoffs|
+-----------------+-------------+---------------+--------------+
|Astoria          |277          |12.48          |644           |
|Brooklyn_Bridge  |3295         |12.02          |3594          |
|Central_Park     |10284        |10.08          |10569         |
|Empire_State     |15808        |10.83          |14158         |
|Harlem           |769          |11.2           |1550          |
|JFK_Airport      |1980         |33.2           |1289          |
|LaGuardia_Airport|1294         |30.13          |953           |
|Long_Island_City |379          |12.84          |654           |
|Times_Square     |12344        |11.07          |11252         |
|Wall_Street      |2869         |14.05          |2889          |
|Williamsburg     |448          |12.57          |1703          |
|Yankee_Stadium   |116          |14.34          |608           |
+-----------

In [68]:
edges = (
    taxi.groupBy(
        F.col("area_pickup").alias("src"),
        F.col("area_dropoff").alias("dst")
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("target_time") / 60, 2).alias("avg_duration_min"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance")
    )
    .dropna(subset=["src", "dst"])
)

edges.cache()
print(f"Edges: {edges.count()}")
edges.orderBy(F.desc("trip_count")).show(10, truncate=False)

Edges: 141
+------------+---------------+----------+--------+----------------+------------+
|src         |dst            |trip_count|avg_fare|avg_duration_min|avg_distance|
+------------+---------------+----------+--------+----------------+------------+
|Empire_State|Empire_State   |6893      |7.24    |8.15            |1.13        |
|Central_Park|Central_Park   |5103      |6.8     |7.08            |1.12        |
|Times_Square|Times_Square   |3901      |7.14    |7.83            |0.99        |
|Empire_State|Times_Square   |3397      |9.5     |11.7            |1.68        |
|Times_Square|Empire_State   |3376      |9.13    |11.0            |1.68        |
|Times_Square|Central_Park   |2915      |9.29    |10.59           |1.84        |
|Central_Park|Times_Square   |2495      |9.86    |11.97           |1.8         |
|Empire_State|Brooklyn_Bridge|1592      |11.49   |13.7            |2.32        |
|Empire_State|Central_Park   |1550      |14.09   |16.93           |3.29        |
|Central_Park|Emp

In [69]:
g = GraphFrame(vertices, edges)

print("GraphFrame created.")
print(f"Vertices : {g.vertices.count()}")
print(f"Edges    : {g.edges.count()}")

GraphFrame created.


/Users/jennykerrcubelo/miniconda3/envs/bda_spark/lib/python3.11/site-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


Vertices : 12
Edges    : 141


The graph has **12 vertices** (NYC hotspot areas) and **141 edges** (unique routes).

With 12 nodes, the maximum possible directed edges without self-loops is 12 × 11 = 132.
The extra 9 edges are self-loops, trips where pickup and dropoff snapped to the same 
hotspot area due to the `match_areas()` function assigning the nearest hotspot to each 
coordinate.

---
## 4. Degree, in-degree, out-degree

- **degree**: total edges (trips in + out)
- **inDegree**: incoming edges (trips arriving)
- **outDegree**: outgoing edges (trips departing)

In [70]:
degrees     = g.degrees
in_degrees  = g.inDegrees
out_degrees = g.outDegrees

degree_summary = (
    g.vertices
    .join(degrees,     on="id", how="left")
    .join(in_degrees,  on="id", how="left")
    .join(out_degrees, on="id", how="left")
    .fillna({"degree": 0, "inDegree": 0, "outDegree": 0})
)
degree_summary.cache()

degree_summary.select(
    "id", "total_pickups", "total_dropoffs",
    "degree", "inDegree", "outDegree"
).orderBy(F.desc("degree")).show(truncate=False)

/Users/jennykerrcubelo/miniconda3/envs/bda_spark/lib/python3.11/site-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+-----------------+-------------+--------------+------+--------+---------+
|id               |total_pickups|total_dropoffs|degree|inDegree|outDegree|
+-----------------+-------------+--------------+------+--------+---------+
|Times_Square     |12344        |11252         |24    |12      |12       |
|Central_Park     |10284        |10569         |24    |12      |12       |
|LaGuardia_Airport|1294         |953           |24    |12      |12       |
|Brooklyn_Bridge  |3295         |3594          |24    |12      |12       |
|Astoria          |277          |644           |24    |12      |12       |
|Harlem           |769          |1550          |24    |12      |12       |
|Empire_State     |15808        |14158         |24    |12      |12       |
|JFK_Airport      |1980         |1289          |24    |12      |12       |
|Williamsburg     |448          |1703          |23    |11      |12       |
|Wall_Street      |2869         |2889          |23    |11      |12       |
|Long_Island_City |379   

Most areas have degree=24 (inDegree=12, outDegree=12), meaning trips exist 
between all area pairs including self-loops. Yankee_Stadium is the exception 
with outDegree=9, reflecting its very low pickup volume (116 trips).

---
## 5. Graph density

In [71]:
num_vertices = g.vertices.count()
num_edges    = g.edges.count()
max_edges    = num_vertices * num_vertices  # n² to account for self-loops
density      = num_edges / max_edges if max_edges > 0 else 0

print(f"Vertices : {num_vertices}")
print(f"Edges    : {num_edges}")
print(f"Max edges: {max_edges}")
print(f"Density  : {density:.4f} ({density*100:.1f}%)")

Vertices : 12
Edges    : 141
Max edges: 144
Density  : 0.9792 (97.9%)


The density formula is adjusted to n² to account for self-loops. A self-loop 
occurs when a trip's pickup and dropoff coordinates both snap to the same 
hotspot area. For example, a trip starting and ending near Times Square. 
These are real trips in the data, not errors. With this adjustment, 97.9% of 
all possible routes between areas (including self-routes) exist in the dataset, 
reflecting the high connectivity of NYC taxi traffic.

---
## 6. PageRank

A zone has high PageRank if many trips arrive from zones that themselves have high PageRank, captures structural importance beyond raw volume.

Parameters: `resetProbability=0.15`, `maxIter=5` 

In [72]:
pagerank_results = g.pageRank(resetProbability=0.15, maxIter=5)

pr_vertices = (
    pagerank_results.vertices
    .select("id", "total_pickups", "total_dropoffs", "pagerank")
    .join(degrees.withColumnRenamed("id", "deg_id"),
          F.col("id") == F.col("deg_id"), how="left").drop("deg_id")
    .join(in_degrees.withColumnRenamed("id", "in_id"),
          F.col("id") == F.col("in_id"), how="left").drop("in_id")
    .fillna(0)
    .cache()
)

print("Top areas by PageRank:")
pr_vertices.orderBy(F.desc("pagerank")).show(truncate=False)

Top areas by PageRank:
+-----------------+-------------+--------------+------------------+------+--------+
|id               |total_pickups|total_dropoffs|pagerank          |degree|inDegree|
+-----------------+-------------+--------------+------------------+------+--------+
|Harlem           |769          |1550          |1.0241820766362055|24    |12      |
|Yankee_Stadium   |116          |608           |1.0241820766362055|21    |12      |
|Central_Park     |10284        |10569         |1.0241820766362055|24    |12      |
|LaGuardia_Airport|1294         |953           |1.0241820766362055|24    |12      |
|JFK_Airport      |1980         |1289          |1.0241820766362055|24    |12      |
|Empire_State     |15808        |14158         |1.0241820766362055|24    |12      |
|Times_Square     |12344        |11252         |1.0241820766362055|24    |12      |
|Brooklyn_Bridge  |3295         |3594          |1.0241820766362055|24    |12      |
|Astoria          |277          |644           |1.024

The PageRank scores reveal a nearly uniform distribution across all 12 NYC zones, with two distinct groups:

- **Top group (PageRank ≈ 1.024):** 9 areas including `Times Square`, `Empire State`, `JFK Airport`, and `Central Park`. These all share inDegree = 12, meaning they receive trips from every other zone in the graph.

- **Bottom group (PageRank ≈ 0.927):** `Williamsburg`, `Wall Street`, and `Long Island City`, each with inDegree = 11, indicating one zone does not send trips directly to them.

The near-uniform scores are expected given the small, highly interconnected graph (12 zones, ~97.9% density). With almost every zone connected to every other, structural importance levels out. In this context, PageRank effectively acts as an indicator for in-degree, rewarding zones that are reachable from all other areas.

---
## 7. Breadth-First Search (BFS)


What is the path from the least active area to the most active area ?

Example: `Yankee Stadium` (least pick ups) to `Empire State` (most pick ups):

In [73]:
bfs_result = g.bfs(
    fromExpr="id = 'Yankee_Stadium'",
    toExpr="id = 'Empire_State'"
)
bfs_result.show(truncate=False)

+---------------------------------+-----------------------------------------------------+-----------------------------------+
|from                             |e0                                                   |to                                 |
+---------------------------------+-----------------------------------------------------+-----------------------------------+
|{Yankee_Stadium, 116, 14.34, 608}|{Yankee_Stadium, Empire_State, 6, 30.58, 30.45, 6.55}|{Empire_State, 15808, 10.83, 14158}|
+---------------------------------+-----------------------------------------------------+-----------------------------------+



BFS finds the shortest path from Yankee Stadium (the least active area, 
116 pickups) to Empire State (the most active area, 15808 pickups). 
Despite the large difference in activity, they are directly connected 
with 6 trips averaging $30.58 fare and 30 minutes travel time.

---
## 8. Label Propagation

Each node adopts the label of the majority of its neighbours at each iteration. Nodes sharing a label at convergence form a **community**, areas with strong mutual trip flows.

In [74]:
lpa_results = g.labelPropagation(maxIter=5).cache()

community_sizes = (
    lpa_results.groupBy("label")
               .agg(F.count("*").alias("community_size"))
               .orderBy(F.desc("community_size"))
)
print("Community size distribution:")
community_sizes.show(truncate=False)

print("\nArea → community:")
lpa_results.select("id", "label").orderBy("label").show(truncate=False)

Community size distribution:
+------------+--------------+
|label       |community_size|
+------------+--------------+
|111669149696|12            |
+------------+--------------+


Area → community:
+-----------------+------------+
|id               |label       |
+-----------------+------------+
|Times_Square     |111669149696|
|Central_Park     |111669149696|
|Yankee_Stadium   |111669149696|
|LaGuardia_Airport|111669149696|
|Brooklyn_Bridge  |111669149696|
|Empire_State     |111669149696|
|Wall_Street      |111669149696|
|JFK_Airport      |111669149696|
|Harlem           |111669149696|
|Long_Island_City |111669149696|
|Astoria          |111669149696|
|Williamsburg     |111669149696|
+-----------------+------------+



Label propagation assigns all 12 areas to a single community, confirming 
that the NYC taxi network is fully interconnected with no distinct 
subgroups. Every area exchanges trips with every other area.

---
## 9. Motif finding


In [75]:
hub_motif = (
    g.find("(a)-[e1]->(b); (a)-[e2]->(c)")
     .filter("b.id != c.id AND b.id != a.id AND c.id != a.id")
     .groupBy(F.col("a.id").alias("hub_area"))
     .agg(F.countDistinct(F.col("b.id"), F.col("c.id")).alias("destination_pairs"))
     .orderBy(F.desc("destination_pairs"))
)
print("Motif 1 — Distributor areas:")
hub_motif.show(truncate=False)

Motif 1 — Distributor areas:
+-----------------+-----------------+
|hub_area         |destination_pairs|
+-----------------+-----------------+
|Long_Island_City |110              |
|Times_Square     |110              |
|Wall_Street      |110              |
|Empire_State     |110              |
|Astoria          |110              |
|Williamsburg     |110              |
|Harlem           |110              |
|Brooklyn_Bridge  |110              |
|JFK_Airport      |110              |
|LaGuardia_Airport|110              |
|Central_Park     |110              |
|Yankee_Stadium   |56               |
+-----------------+-----------------+



**Motif 1** identifies distributor zones, areas that send trips to at least two distinct destinations simultaneously.

All areas act as distributors sending trips to 110 destination pairs, 
confirming the full connectivity of the network. Yankee Stadium is the 
only exception with 56 destination pairs, consistent with its low pickup 
volume of 116 trips, not enough traffic to cover all possible route 
combinations.



In [76]:
convergence_motif = (
    g.find("(a)-[e1]->(c); (b)-[e2]->(c)")
     .filter("a.id != b.id AND a.id != c.id AND b.id != c.id")
     .groupBy(F.col("c.id").alias("convergence_area"))
     .agg(F.countDistinct(F.col("a.id"), F.col("b.id")).alias("origin_pairs"))
     .orderBy(F.desc("origin_pairs"))
)
print("Motif 2 — Convergence areas:")
convergence_motif.show(truncate=False)

Motif 2 — Convergence areas:
+-----------------+------------+
|convergence_area |origin_pairs|
+-----------------+------------+
|Times_Square     |110         |
|Empire_State     |110         |
|Astoria          |110         |
|Harlem           |110         |
|Brooklyn_Bridge  |110         |
|JFK_Airport      |110         |
|LaGuardia_Airport|110         |
|Central_Park     |110         |
|Yankee_Stadium   |110         |
|Long_Island_City |90          |
|Wall_Street      |90          |
|Williamsburg     |90          |
+-----------------+------------+



**Motif 2** identifies convergence zones, areas that receive trips from at least two distinct origins simultaneously.

Most areas receive trips from 110 origin pairs, confirming they are 
well-reached from all parts of the network. Long Island City, Wall Street 
and Williamsburg are the exceptions with 90 origin pairs, consistent 
with their lower inDegree (11) seen in the degree analysis, meaning one 
area does not send trips to them.

---
## 10. Weighted flow analysis

Unlike unweighted graphs, our edges carry `trip_count`, `avg_fare`, `avg_duration_min` and `avg_distance`. We use these to identify the most important and most expensive routes.

In [77]:
print("Top 15 most frequent routes:")
g.edges.orderBy(F.desc("trip_count")).show(15, truncate=False)

print("Top 10 most expensive routes (min 50 trips):")
g.edges.filter(F.col("trip_count") >= 50)\
       .orderBy(F.desc("avg_fare"))\
       .show(10, truncate=False)

Top 15 most frequent routes:
+---------------+---------------+----------+--------+----------------+------------+
|src            |dst            |trip_count|avg_fare|avg_duration_min|avg_distance|
+---------------+---------------+----------+--------+----------------+------------+
|Empire_State   |Empire_State   |6893      |7.24    |8.15            |1.13        |
|Central_Park   |Central_Park   |5103      |6.8     |7.08            |1.12        |
|Times_Square   |Times_Square   |3901      |7.14    |7.83            |0.99        |
|Empire_State   |Times_Square   |3397      |9.5     |11.7            |1.68        |
|Times_Square   |Empire_State   |3376      |9.13    |11.0            |1.68        |
|Times_Square   |Central_Park   |2915      |9.29    |10.59           |1.84        |
|Central_Park   |Times_Square   |2495      |9.86    |11.97           |1.8         |
|Empire_State   |Brooklyn_Bridge|1592      |11.49   |13.7            |2.32        |
|Empire_State   |Central_Park   |1550      |14.

***Most frequent routes***: The top routes are dominated by short trips within the Midtown core. Notably, the three most frequent routes are self-loops (Empire State → Empire State, Central Park → Central Park, Times Square → Times Square), suggesting many trips start and end within the same hotspot zone. This is a consequence of how zones were defined, short rides that don't cross zone boundaries get assigned the same pickup and dropoff area. The busiest cross-area route is Empire State ↔ Times Square, with over 6,700 trips in both directions combined.

***Most expensive routes***: Airport connections dominate the top fares, with JFK → Wall Street averaging $51.59, the most expensive route in the dataset. All top 10 routes involve either JFK Airport or LaGuardia Airport, reflecting the fixed flat-rate and long-distance surcharges applied to airport rides. Trips to/from JFK average around $48–52 with journey times of 38–44 minutes, consistent with the NYC flat fare policy for JFK rides.

In [78]:
flow_pd = g.edges.select("src", "dst", "trip_count").toPandas()
all_areas = sorted(set(flow_pd["src"]).union(set(flow_pd["dst"])))

pivot = (
    flow_pd.pivot(index="src", columns="dst", values="trip_count")
           .fillna(0)
           .reindex(index=all_areas, columns=all_areas, fill_value=0)
)

fig = px.imshow(
    pivot,
    title="Origin-Destination Trip Flow Matrix",
    labels={"x": "Dropoff Area", "y": "Pickup Area", "color": "Trip Count"},
    color_continuous_scale="YlOrRd",
    text_auto=True
)
fig.update_layout(height=600)
fig.show()

The matrix confirms the dominance of routes between Manhattan central areas. 
Empire State and Times Square exchange the highest trip volumes, reflecting their central position in NYC. Yankee Stadium and Astoria show consistently low flows across all routes, consistent with their low pickup volumes seen in the degree analysis.

## 11. NYC Flow Map

Visualization of the taxi trip flow between NYC zones on an interactive map. Each node represents a hotspot area, with node size proportional to total trip volume. Only routes with at least 200 trips are shown.

In [79]:
import plotly.graph_objects as go

# Define hotspot coordinates from NYC_HOTSPOTS dict
hotspot_coords = {name: coords for name, coords in NYC_HOTSPOTS.items()}

# Total trips per area (pickups + dropoffs) as a pandas Series
node_totals = (
    flow_pd.groupby("src")["trip_count"].sum()
    .add(flow_pd.groupby("dst")["trip_count"].sum(), fill_value=0)
)

fig = go.Figure()

# Edges
edge_lons, edge_lats = [], []
top_edges = flow_pd[flow_pd["trip_count"] >= 200].copy()

for _, row in top_edges.iterrows():
    src, dst = row["src"], row["dst"]
    if src == dst or src not in hotspot_coords or dst not in hotspot_coords:
        continue
    lon0, lat0 = hotspot_coords[src]
    lon1, lat1 = hotspot_coords[dst]
    edge_lons += [lon0, lon1, None]
    edge_lats += [lat0, lat1, None]

fig.add_trace(go.Scattermap(
    lon=edge_lons,
    lat=edge_lats,
    mode="lines",
    line=dict(width=1.5, color="rgba(30, 100, 200, 0.4)"),
    hoverinfo="skip",
    showlegend=False
))

# Nodes
active_nodes = set(top_edges["src"]) | set(top_edges["dst"])

names = [a for a in hotspot_coords if a in active_nodes]
lons  = [hotspot_coords[a][0] for a in names]
lats  = [hotspot_coords[a][1] for a in names]
sizes = [8 + 18 * (node_totals.get(n, 0) / node_totals.max()) for n in names]

fig.add_trace(go.Scattermap(
    lon=lons, lat=lats,
    mode="markers",
    marker=dict(size=sizes, color="#E63946", opacity=0.9),
    hovertext=[f"<b>{n.replace('_', ' ')}</b><br>Total trips: {int(node_totals.get(n, 0)):,}" for n in names],
    hoverinfo="text",
    showlegend=False
))

# Layout
fig.update_layout(
    map=dict(
        style="carto-positron",
        center=dict(lat=40.73, lon=-73.96),
        zoom=11.5
    ),
    title=dict(
        text="NYC Taxi Flow Map — Trip Routes Between Zones",
        x=0.5,
        font=dict(size=16)
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    height=650
)

fig.show()

In [80]:
#cleanup
taxi.unpersist()
edges.unpersist()
degree_summary.unpersist()
pr_vertices.unpersist()
lpa_results.unpersist()
spark.stop()
print("Spark stopped.")

Spark stopped.
